In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import gc

In [4]:
gc.collect()
df1 = pd.read_csv("./df_knni_rounded.csv", sep='\t').copy().reset_index(drop = True)
df1

,manufacturing_year,battery_capacity_kwh,odometer_km,vehicle_age_years,cycle_count,battery_health_percent,state_of_charge,depth_of_discharge,state_of_health,cell_voltage_avg,...,predicted_remaining_life_cycles,battery_failure,Vehicle_Brand_Num_Value,Vehicle_Model_Num_Value,Battery_Manufacturer_Num_Value,Terrain_Type_Num_Value,Battery_Chemistry_Num_Value,Drive_Type_Num_Value,Fleet_Or_Private_Num_Value,Vehicle_Type_Num_Value
0,2019.0,82.54,98163.0,6.94,250.0,73.61,85.230,26.34,72.93,3.60246,...,401.0,0.0,12.0,24.0,1.0,2.0,1.0,2.0,1.0,2.0
1,2018.0,64.17,113600.0,7.79,256.0,85.77,49.120,44.83,86.71,3.36580,...,1094.0,0.0,0.0,49.0,8.0,3.0,4.0,0.0,1.0,3.0
2,2018.0,90.92,242253.0,7.85,556.0,72.37,40.086,33.65,72.17,3.41520,...,645.0,0.0,17.0,48.0,3.0,2.0,4.0,2.0,0.0,6.0
3,2022.0,47.35,83700.0,6.44,353.0,85.11,54.320,37.73,82.88,3.42020,...,867.0,0.0,18.0,18.0,1.0,3.0,4.0,0.0,1.0,1.0
4,2015.0,68.04,88784.0,11.13,287.0,87.40,62.730,65.29,86.17,3.45560,...,1090.0,0.0,16.0,29.0,0.0,0.0,4.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,2016.0,45.99,182311.0,10.11,817.0,86.91,30.510,42.42,84.85,2.89010,...,2691.0,0.0,2.0,40.0,1.0,2.0,0.0,1.0,1.0,1.0
199996,2015.0,67.65,275317.0,11.06,840.0,65.08,77.610,30.76,65.08,3.57070,...,420.0,0.0,8.0,0.0,4.0,3.0,4.0,0.0,1.0,0.0
199997,2019.0,98.08,158747.0,4.89,336.0,86.08,44.590,24.64,84.90,2.98750,...,3148.0,0.0,2.0,40.0,0.0,4.0,0.0,1.0,0.0,2.0
199998,2016.0,51.49,95122.0,10.06,392.0,76.61,80.700,41.93,75.50,3.56980,...,797.0,0.0,16.0,30.0,7.0,0.0,4.0,1.0,1.0,1.0


In [11]:
class Random_Forest_Class(object):
    def __init__(self, dataset, n_estimators : int, criterion : str, max_depth : int, min_samples_split : int or float, 
                 min_samples_leaf : int or float, min_weight_fraction_leaf : float, 
                 max_features : str, max_leaf_nodes : int, min_impurity_decrease : float, 
                 bootstrap : bool, oob_score : bool, n_jobs : int, random_state : int or None, 
                 verbose : int, warm_start : bool, 
                 class_weight : str, ccp_alpha : float, max_samples : int or float, monotonic_cst):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.n_estimators = n_estimators
        self.criterion = criterion
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.min_weight_fraction_leaf = min_weight_fraction_leaf
        self.max_features = max_features
        self.max_leaf_nodes = max_leaf_nodes
        self.min_impurity_decrease = min_impurity_decrease
        self.bootstrap = bootstrap
        self.oob_score = oob_score
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.verbose = verbose
        self.warm_start = warm_start
        self.class_weight = class_weight
        self.ccp_alpha = ccp_alpha
        self.max_samples = max_samples
        self.monotonic_cst = monotonic_cst
        
    def train_split(self):

        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        from sklearn.model_selection import train_test_split
        df4 = self.dataset.copy().reset_index(drop = True)
        Y_col = 'Vehicle_Brand_Num_Value'
        X_cols = df4.loc[:, df4.columns != Y_col].columns
        # Split into 80% training and 20% testing
        X_train, X_test, y_train, y_test = train_test_split(
            df4[X_cols], df4[Y_col], test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def ML_fitting(self):
        global rf_fit_global

        rf = RandomForestClassifier(n_estimators = self.n_estimators, criterion = self.criterion, 
                                    max_depth = self.max_depth, min_samples_split = self.min_samples_split, 
                                    min_samples_leaf = self.min_samples_leaf, 
                                    min_weight_fraction_leaf = self.min_weight_fraction_leaf, 
                                    max_features = self.max_features, max_leaf_nodes = self.max_leaf_nodes, 
                                    min_impurity_decrease = self.min_impurity_decrease, 
                                    bootstrap = self.bootstrap, oob_score = self.oob_score, 
                                    n_jobs = self.n_jobs, random_state = self.random_state, 
                                    verbose = self.verbose, warm_start = self.warm_start, 
                                    class_weight = self.class_weight, ccp_alpha = self.ccp_alpha, 
                                    max_samples = self.max_samples, monotonic_cst = self.monotonic_cst)
        rf_fit = rf.fit(X_train_global, y_train_global)
        rf_fit_global = rf_fit

    def ML_prediction(self):
        global rf_y_predict_global

        gc.collect()
        rf_y_predict = rf_fit_global.predict(X_test_global)
        rf_y_predict_global = rf_y_predict

    def evaluation(self):

        from sklearn.metrics import accuracy_score
        accuracy = accuracy_score(y_test_global, rf_y_predict_global)
        print("Accuracy:", accuracy)

        from sklearn.metrics import balanced_accuracy_score
        balanced_accuracy_score = balanced_accuracy_score(y_test_global, rf_y_predict_global)
        print("balanced_accuracy_score:", balanced_accuracy_score)

        from sklearn.metrics import top_k_accuracy_score
        top_k_accuracy_score = top_k_accuracy_score(y_test_global,rf_fit_global.predict_proba(X_test_global), k = 1)
        print("top_k_accuracy_score:", top_k_accuracy_score)

        from sklearn.metrics import average_precision_score
        average_precision_score = average_precision_score(y_test_global,rf_fit_global.predict_proba(X_test_global))
        print("average_precision_score:", average_precision_score)

        from sklearn.metrics import brier_score_loss
        brier_score_loss = brier_score_loss(y_test_global,rf_fit_global.predict_proba(X_test_global))
        print("brier_score_loss:", brier_score_loss)

        from sklearn.metrics import f1_score
        f1_score = f1_score(y_test_global, rf_y_predict_global, average='macro')
        print("f1_score_macro:", f1_score)
        from sklearn.metrics import f1_score
        f1_score_1 = f1_score(y_test_global, rf_y_predict_global, average='micro')
        print("f1_score_micro:", f1_score_1)
        f1_score_2 = f1_score(y_test_global, rf_y_predict_global, average='weighted')
        print("f1_score_weighted:", f1_score_2)
        f1_score_3 = f1_score(y_test_global, rf_y_predict_global, average=None)
        print("f1_score_None:", f1_score_3)
        #f1_score_4 = f1_score(y_test_global, rf_y_predict_global, average='binary')
        #print("f1_score_binary:", f1_score_4)

        from sklearn.metrics import log_loss
        log_loss = log_loss(y_test_global,rf_fit_global.predict_proba(X_test_global))
        print("log_loss:", log_loss)

        from sklearn.metrics import precision_score
        precision_score_macro = precision_score(y_test_global, rf_y_predict_global, average='macro')
        print("precision_score_macro:", precision_score_macro)
        precision_score_micro = precision_score(y_test_global, rf_y_predict_global, average='micro')
        print("precision_score_micro:", precision_score_micro)
        precision_score_weighted = precision_score(y_test_global, rf_y_predict_global, average='weighted')
        print("precision_score_weighted:", precision_score_weighted)
        #precision_score_binary = precision_score(y_test_global, rf_y_predict_global, average='binary')
        #print("precision_score_binary:", precision_score_binary)
        precision_score_None = precision_score(y_test_global, rf_y_predict_global, average=None)
        print("precision_score_None:", precision_score_None)
        precision_score_None_ZD_1 = precision_score(y_test_global, rf_y_predict_global, average=None, zero_division=1)
        print("precision_score_None_ZD_1:", precision_score_None_ZD_1)
        precision_score_None_ZD_NAN = precision_score(y_test_global, rf_y_predict_global, average=None, zero_division=np.nan)
        print("precision_score_None_ZD_NAN:", precision_score_None_ZD_NAN)

        from sklearn.metrics import recall_score
        recall_score_macro = recall_score(y_test_global, rf_y_predict_global, average='macro')
        print("recall_score_macro:", recall_score_macro)
        recall_score_micro = recall_score(y_test_global, rf_y_predict_global, average='micro')
        print("recall_score_micro:", recall_score_micro)
        recall_score_weighted = recall_score(y_test_global, rf_y_predict_global, average='weighted')
        print("recall_score_weighted:", recall_score_weighted)
        #recall_score_binary = recall_score(y_test_global, rf_y_predict_global, average='binary')
        #print("recall_score_binary:", recall_score_binary)
        recall_score_None = recall_score(y_test_global, rf_y_predict_global, average=None)
        print("recall_score_None:", recall_score_None)
        recall_score_None_ZD_1 = recall_score(y_test_global, rf_y_predict_global, average=None, zero_division=1)
        print("recall_score_None_ZD_1:", recall_score_None_ZD_1)
        recall_score_None_ZD_NAN = recall_score(y_test_global, rf_y_predict_global, average=None, zero_division=np.nan)
        print("recall_score_None_ZD_NAN:", recall_score_None_ZD_NAN)

        from sklearn.metrics import jaccard_score
        jaccard_score_macro = jaccard_score(y_test_global, rf_y_predict_global, average='macro')
        print("jaccard_score_macro:", jaccard_score_macro)
        jaccard_score_micro = jaccard_score(y_test_global, rf_y_predict_global, average='micro')
        print("jaccard_score_micro:", jaccard_score_micro)
        jaccard_score_weighted = jaccard_score(y_test_global, rf_y_predict_global, average='weighted')
        print("jaccard_score_weighted:", jaccard_score_weighted)
        #jaccard_score_binary = jaccard_score(y_test_global, rf_y_predict_global, average='binary')
        #print("jaccard_score_binary:", jaccard_score_binary)
        jaccard_score_None = jaccard_score(y_test_global, rf_y_predict_global, average=None)
        print("jaccard_score_None:", jaccard_score_None)

        from sklearn.metrics import roc_auc_score
        roc_auc_score = roc_auc_score(y_test_global, rf_fit_global.predict_proba(X_test_global), multi_class='ovr')
        print('roc_auc_score:', roc_auc_score)

        from sklearn.metrics import roc_auc_score
        roc_auc_score_macro = roc_auc_score(y_test_global, rf_fit_global.predict_proba(X_test_global), average='macro',
                                           multi_class='ovr')
        print("roc_auc_score_macro:", roc_auc_score_macro)
        roc_auc_score_micro = roc_auc_score(y_test_global, rf_fit_global.predict_proba(X_test_global), average='micro',
                                           multi_class='ovr')
        print("roc_auc_score_micro:", roc_auc_score_micro)
        roc_auc_score_weighted = roc_auc_score(y_test_global, rf_fit_global.predict_proba(X_test_global), average='weighted',
                                              multi_class='ovr')
        print("roc_auc_score_weighted:", roc_auc_score_weighted)
        roc_auc_score_None = roc_auc_score(y_test_global, rf_fit_global.predict_proba(X_test_global), average=None,
                                          multi_class='ovr')
        print("roc_auc_score_None:", roc_auc_score_None)

        from sklearn.metrics import d2_log_loss_score
        d2_log_loss_score = d2_log_loss_score(y_test_global, rf_fit_global.predict_proba(X_test_global),
                                              sample_weight=None, labels=None)
        print("d2_log_loss_score:", d2_log_loss_score)

        from sklearn.metrics import d2_brier_score
        d2_brier_score = d2_brier_score(y_test_global, rf_fit_global.predict_proba(X_test_global), 
                                        sample_weight=None, labels=None)
        print("d2_brier_score:", d2_brier_score)
        

    def main(self):
        gc.collect()
        self.train_split()
        self.ML_fitting()
        self.ML_prediction()
        self.evaluation()
        gc.collect()
        

In [12]:
rf_1 = Random_Forest_Class(df1, 100, 'gini', None, 5, 5, 0.25, 'log2', None, 0.5, True, True, None,
                          4, 2, False, 'balanced', 1.2, None, None)
rf_1.main()

building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100


[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    3.5s


building tree 42 of 100
building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80 of 100
building tree 81 of 100
building tree 82 of 100
building tree 83

[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    8.7s finished
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.2s finished


Accuracy: 0.0373
balanced_accuracy_score: 0.05


[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.3s finished


... (17 outputs left out)

[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.3s finished


d2_log_loss_score: -0.03180523481051756


[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 100 out of 100 | elapsed:    0.3s finished


d2_brier_score: -0.010979912622434584


In [13]:
rf_fit_global

,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.25
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.5
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",4
,"verbose verbose: int, default=0Controls the verbosity when fitting and predicting.",2
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of 